# ☕ POSTNOW — Gemma 4 Fine-tuning Notebook

Fine-tunes **Google Gemma 4** (`gemma-4-E4B-it`) on the POSTNOW 2,500-caption bilingual dataset using **QLoRA** (4-bit quantisation + LoRA adapters).

---

### Before you start
1. **Runtime → Change runtime type → T4 GPU** (free tier is enough)
2. Make sure you accepted the Gemma 4 license at [huggingface.co/google/gemma-4-E4B-it](https://huggingface.co/google/gemma-4-E4B-it)
3. Upload `postnow_training_data_2500.json` using the **Files panel** (folder icon on the left)

---
| Step | Cell | Time |
|------|------|------|
| Install packages | §1 | ~3 min |
| Login to HuggingFace | §2 | 1 min |
| Upload dataset | §3 | 1 min |
| Smoke test (20 steps) | §4 | ~3 min |
| Full training (3 epochs) | §5 | ~60-90 min |
| Test inference | §6 | ~2 min |
| Download adapter | §7 | 1 min |

## §1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q \
    transformers>=4.44.0 \
    peft>=0.12.0 \
    trl>=0.10.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.33.0 \
    datasets>=2.20.0 \
    sentencepiece

print("✅ All packages installed")

In [ ]:
# Verify GPU is available
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU detected: {gpu}")
    print(f"   VRAM: {vram:.1f} GB")
else:
    print("❌ No GPU detected — go to Runtime → Change runtime type → T4 GPU")

## §2 — Login to HuggingFace

Get your token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (read access is enough).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## §3 — Upload & Verify Dataset

Upload `postnow_training_data_2500.json` via the **Files panel** (📁 icon on the left sidebar), then run the cell below to verify it loaded correctly.

In [ ]:
import json
from pathlib import Path

DATA_FILE = "/content/postnow_training_data_2500.json"

if not Path(DATA_FILE).exists():
    print("❌ File not found. Upload postnow_training_data_2500.json first.")
else:
    with open(DATA_FILE) as f:
        data = json.load(f)
    captions = data.get("captions", [])
    cats = {}
    for c in captions:
        cats[c["category"]] = cats.get(c["category"], 0) + 1

    print(f"✅ Dataset loaded: {len(captions)} captions")
    print("   Category breakdown:")
    for cat, count in cats.items():
        print(f"     {cat:<22} {count}")

    # Preview one entry
    sample = captions[0]
    print(f"\n📝 Sample entry (id={sample['id']}, {sample['category']}):")
    print(f"   EN: {sample['caption']['en'][:100]}...")
    print(f"   KH: {sample['caption']['km'][:100]}...")

## §4 — Configuration & Dataset Preparation

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Change MODEL_ID here if you want a different Gemma 4 variant:
#   google/gemma-4-E2B-it   → 2B active params, fastest, lowest VRAM
#   google/gemma-4-E4B-it   → 4B active params, best balance for T4  ← DEFAULT
#   google/gemma-4-26B-A4B-it → needs A100
#   google/gemma-4-31B-it   → needs A100 80GB

MODEL_ID      = "google/gemma-4-E4B-it"
DATA_FILE     = "/content/postnow_training_data_2500.json"
OUTPUT_DIR    = "/content/gemma_postnow"
ADAPTER_DIR   = f"{OUTPUT_DIR}/lora_adapter"

# Training hyperparameters
EPOCHS        = 3
BATCH_SIZE    = 2      # per-device (T4 safe)
GRAD_ACCUM    = 8      # effective batch = 2 × 8 = 16
LR            = 2e-4
MAX_SEQ_LEN   = 768
LORA_R        = 16
LORA_ALPHA    = 32
VAL_RATIO     = 0.10

print(f"Model       : {MODEL_ID}")
print(f"Epochs      : {EPOCHS}")
print(f"Eff. batch  : {BATCH_SIZE * GRAD_ACCUM}")
print(f"LR          : {LR}")
print(f"LoRA r/α    : {LORA_R}/{LORA_ALPHA}")
print(f"Max seq len : {MAX_SEQ_LEN}")
print(f"Output      : {OUTPUT_DIR}")

In [ ]:
# ── System prompt & formatting helpers ────────────────────────────────────────

SYSTEM_PROMPT = (
    "You are a bilingual social media copywriter for Cambodian coffee shops. "
    "Generate creative, natural-sounding captions in both English and Khmer. "
    "Use mixed Khmer-English for coffee terms (Latte, Cold Brew, Espresso…). "
    "Never translate word-for-word — always adapt the tone for a Cambodian audience."
)

def build_instruction(entry):
    meta = entry.get("metadata", {})
    cat  = entry.get("category", "general")
    lines = [f"Category: {cat.replace('_', ' ').title()}",
             f"Product: {meta.get('product', '')}"]
    if meta.get("promotion"):
        lines.append(f"Promotion: {meta['promotion']}")
    lines += [f"Mood: {meta.get('mood', '')}",
              f"Occasion: {meta.get('occasion', '')}",
              f"Target audience: {meta.get('target_audience', '')}"]
    return "\n".join(lines)

def build_response(entry):
    cap  = entry.get("caption", {})
    tags = entry.get("hashtags", {})
    en_tags = " ".join(tags.get("en", []))
    km_tags = " ".join(tags.get("km", []))
    return (
        f"[EN]\n{cap.get('en', '')}\n\n"
        f"[KH]\n{cap.get('km', '')}\n\n"
        f"[TAGS]\n{(en_tags + ' ' + km_tags).strip()}"
    )

def format_chat(tokenizer, entry):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Generate a coffee shop social media caption:\n\n{build_instruction(entry)}"},
        {"role": "assistant", "content": build_response(entry)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

print("✅ Formatting helpers defined")

In [ ]:
# ── Load tokenizer & prepare HuggingFace Dataset ──────────────────────────────
import random
from datasets import Dataset
from transformers import AutoTokenizer

print(f"Loading tokenizer: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("✅ Tokenizer loaded")

# Build chat-formatted texts
print("Formatting dataset...")
with open(DATA_FILE) as f:
    raw = json.load(f)
entries = raw.get("captions", [])

random.seed(42)
random.shuffle(entries)

texts = [{"text": format_chat(tokenizer, e)} for e in entries]

n_val   = max(1, int(len(texts) * VAL_RATIO))
train_t = texts[n_val:]
val_t   = texts[:n_val]

train_ds = Dataset.from_list(train_t)
val_ds   = Dataset.from_list(val_t)

print(f"✅ Dataset ready → {len(train_ds)} train / {len(val_ds)} val")

# Show a formatted sample
print("\n📝 Example formatted prompt (first 600 chars):")
print(train_ds[0]["text"][:600])

## §5 — Load Model + Apply QLoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ── 4-bit QLoRA config ────────────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {MODEL_ID} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print("✅ Base model loaded")

# ── LoRA config ───────────────────────────────────────────────────────────────
# target_modules="all-linear" auto-detects all nn.Linear layers,
# which correctly covers both attention AND MoE expert layers in Gemma 4.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    inference_mode=False,
)
model = get_peft_model(model, lora_config)

print("\n✅ LoRA applied:")
model.print_trainable_parameters()

## §6 — Smoke Test (20 Steps)

Run this **before** full training to verify everything works end-to-end (~3 minutes).

In [ ]:
from trl import SFTTrainer, SFTConfig
from pathlib import Path

smoke_dir = "/content/smoke_test_output"

smoke_config = SFTConfig(
    output_dir=smoke_dir,
    max_steps=20,                         # only 20 steps
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    logging_steps=5,
    report_to="none",
    optim="paged_adamw_8bit",
    eval_strategy="no",
    save_strategy="no",
    dataset_num_proc=1,
)

smoke_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=smoke_config,
    train_dataset=train_ds,
)

print("🚀 Running smoke test (20 steps)...")
smoke_trainer.train()
print("\n✅ Smoke test passed! Safe to run full training.")

## §7 — Full Training

Expected time: **~60–90 minutes on T4** for 3 epochs.

> ⚠️ Make sure Colab is set to not disconnect: keep the browser tab active.

In [ ]:
from trl import SFTTrainer, SFTConfig
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

train_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    optim="paged_adamw_8bit",

    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    logging_steps=20,
    report_to="none",
    seed=42,
    dataset_num_proc=1,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=train_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

eff_batch = BATCH_SIZE * GRAD_ACCUM
print(f"🚀 Starting full training")
print(f"   Epochs        : {EPOCHS}")
print(f"   Train samples : {len(train_ds)}")
print(f"   Val samples   : {len(val_ds)}")
print(f"   Effective batch: {eff_batch}")
print(f"   Steps/epoch   : {len(train_ds) // eff_batch}")
print()

trainer.train()
print("\n✅ Training complete!")

In [ ]:
# ── Save LoRA adapter ─────────────────────────────────────────────────────────
import os

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Save training metadata
meta = {
    "base_model":   MODEL_ID,
    "train_size":   len(train_ds),
    "val_size":     len(val_ds),
    "epochs":       EPOCHS,
    "batch_size":   BATCH_SIZE,
    "grad_accum":   GRAD_ACCUM,
    "lr":           LR,
    "lora_r":       LORA_R,
    "lora_alpha":   LORA_ALPHA,
    "max_seq_len":  MAX_SEQ_LEN,
    "quantisation": "4-bit QLoRA (NF4)",
    "adapter_path": ADAPTER_DIR,
}
with open(f"{OUTPUT_DIR}/training_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

# List saved files
files = list(Path(ADAPTER_DIR).rglob("*"))
total_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6

print(f"✅ LoRA adapter saved → {ADAPTER_DIR}")
print(f"   Files: {len([f for f in files if f.is_file()])}  |  Total size: {total_mb:.1f} MB")
for f in sorted(files):
    if f.is_file():
        print(f"   {f.name}  ({f.stat().st_size/1e6:.2f} MB)")

## §8 — Test Inference

Run a quick test to see the fine-tuned model generate a caption.

In [ ]:
import re

def generate_caption(model, tokenizer, shop_name, aesthetic, colors, promotion, max_new_tokens=500):
    user_msg = (
        f"Generate a coffee shop social media caption:\n\n"
        f"Shop name: {shop_name}\n"
        f"Aesthetic: {aesthetic}\n"
        f"Brand colors: {', '.join(colors)}\n"
        f"Promotion: {promotion}"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# ── Test 1 ────────────────────────────────────────────────────────────────────
print("=" * 60)
print("TEST 1 — Product Launch")
print("=" * 60)
out1 = generate_caption(
    model, tokenizer,
    shop_name  = "Slow Drip Café",
    aesthetic  = "Cozy",
    colors     = ["#C8A27C", "#5A3E2B"],
    promotion  = "Introducing our new Coconut Cold Brew this weekend",
)
print(out1)

# ── Test 2 ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("TEST 2 — Promotion / BOGO")
print("=" * 60)
out2 = generate_caption(
    model, tokenizer,
    shop_name  = "Angkor Espresso",
    aesthetic  = "Bold",
    colors     = ["#E83F6F", "#2274A5"],
    promotion  = "Buy 1 Get 1 Free on all Iced Lattes — Happy Hour 3PM to 5PM",
)
print(out2)

# ── Test 3 ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("TEST 3 — Khmer New Year Special")
print("=" * 60)
out3 = generate_caption(
    model, tokenizer,
    shop_name  = "Mekong Roasters",
    aesthetic  = "Minimalist",
    colors     = ["#F5F5F5", "#1A1A1A"],
    promotion  = "Khmer New Year special: free Iced Coffee for first 50 customers",
)
print(out3)

## §9 — Download Adapter

Zip the adapter folder and download it to your local machine.

In [ ]:
import shutil
from google.colab import files

# Zip the adapter + metadata
zip_path = "/content/gemma_postnow_adapter"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)

zip_size = Path(zip_path + ".zip").stat().st_size / 1e6
print(f"✅ Zipped adapter: {zip_path}.zip  ({zip_size:.1f} MB)")
print("Downloading...")

files.download(zip_path + ".zip")
print("Done! Extract to ml/model/gemma_postnow/ in your POSTNOW project.")

## §10 — (Optional) Push Adapter to HuggingFace Hub

If you want to store the adapter on HuggingFace for easy access later.

In [ ]:
# Optional — uncomment and fill in your repo name

# HF_REPO = "your-username/postnow-gemma4-lora"   # ← change this
# 
# trainer.model.push_to_hub(HF_REPO, private=True)
# tokenizer.push_to_hub(HF_REPO, private=True)
# 
# print(f"✅ Adapter pushed to https://huggingface.co/{HF_REPO}")

print("(Skipped — uncomment the lines above if you want to push to HF Hub)")

---

## Summary

| Item | Value |
|------|-------|
| Base model | `google/gemma-4-E4B-it` |
| Fine-tuning method | QLoRA (4-bit NF4 + LoRA rank 16) |
| Dataset | 2,250 train / 250 val captions |
| Adapter size | ~50–100 MB |
| Output | `[EN]` + `[KH]` + `[TAGS]` |

### Using the adapter in your POSTNOW backend

1. Extract the zip to `ml/model/gemma_postnow/`
2. In `backend/.env` set:
   ```
   USE_LOCAL_MODEL=true
   LOCAL_MODEL_PATH=ml/model/gemma_postnow/lora_adapter
   ```
3. Update `backend/routes/generate.py` to import `infer_gemma` instead of `infer`
4. Test with:
   ```bash
   python ml/infer_gemma.py --adapter_path ml/model/gemma_postnow/lora_adapter
   ```